In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import os
os.listdir("/kaggle/input")

['competitions']

In [2]:
import pandas as pd
train = pd.read_csv("/kaggle/input/titanic/train.csv")
train.head()

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/titanic/train.csv'

In [3]:
import os
os.listdir("/kaggle/input/competitions")

['titanic']

In [4]:
import pandas as pd
train = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [5]:
train.shape

(891, 12)

In [7]:
train.info(verbose=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [6]:
train.isna().sum().sort_values(ascending=False)

Cabin          687
Age            177
Embarked         2
PassengerId      0
Name             0
Pclass           0
Survived         0
Sex              0
Parch            0
SibSp            0
Fare             0
Ticket           0
dtype: int64

In [8]:
train.groupby("Sex")["Survived"].mean().sort_values(ascending=False)

Sex
female    0.742038
male      0.188908
Name: Survived, dtype: float64

In [9]:
train.groupby("Pclass")["Survived"].mean()

Pclass
1    0.629630
2    0.472826
3    0.242363
Name: Survived, dtype: float64

In [10]:
train["Embarked"].value_counts(dropna=False)

Embarked
S      644
C      168
Q       77
NaN      2
Name: count, dtype: int64

In [11]:
train.groupby("Embarked")["Survived"].mean()

Embarked
C    0.553571
Q    0.389610
S    0.336957
Name: Survived, dtype: float64

In [12]:
pd.crosstab(train["Embarked"], train["Pclass"], normalize="index")

Pclass,1,2,3
Embarked,,,
C,0.505952,0.101190,0.392857
Q,0.025974,0.038961,0.935065
S,0.197205,0.254658,0.548137


In [13]:
train.groupby(["Embarked", "Pclass"])["Survived"].mean()

Embarked  Pclass
C         1         0.694118
          2         0.529412
          3         0.378788
Q         1         0.500000
          2         0.666667
          3         0.375000
S         1         0.582677
          2         0.463415
          3         0.189802
Name: Survived, dtype: float64

In [14]:
import pandas as pd

train["AgeBin"] = pd.cut(train["Age"], bins=[0, 12, 18, 35, 50, 80])
train.groupby("AgeBin")["Survived"].mean()

/tmp/ipykernel_55/2332343829.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  train.groupby("AgeBin")["Survived"].mean()


AgeBin
(0, 12]     0.579710
(12, 18]    0.428571
(18, 35]    0.382682
(35, 50]    0.398693
(50, 80]    0.343750
Name: Survived, dtype: float64

In [15]:
train.groupby(["Sex", "Pclass"])["Survived"].mean()

Sex     Pclass
female  1         0.968085
        2         0.921053
        3         0.500000
male    1         0.368852
        2         0.157407
        3         0.135447
Name: Survived, dtype: float64

In [16]:
train.groupby(["Sex", "Pclass"])["Survived"].mean()

Sex     Pclass
female  1         0.968085
        2         0.921053
        3         0.500000
male    1         0.368852
        2         0.157407
        3         0.135447
Name: Survived, dtype: float64

In [17]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report

# 1) Load data
train = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")

# 2) Target + features
y = train["Survived"]

# Minimal strong baseline feature set (skip Cabin/Name/Ticket for now)
feature_cols = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]
X = train[feature_cols].copy()

# 3) Split (stratify keeps class balance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4) Preprocessing
numeric_features = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
categorical_features = ["Sex", "Embarked"]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

# 5) Model
clf = LogisticRegression(max_iter=2000)

model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", clf)
])

# 6) Train
model.fit(X_train, y_train)

# 7) Predict + evaluate
pred = model.predict(X_test)
proba = model.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, pred)
auc = roc_auc_score(y_test, proba)
cm = confusion_matrix(y_test, pred)

print("Accuracy:", acc)
print("ROC-AUC:", auc)
print("\nConfusion matrix:\n", cm)
print("\nClassification report:\n", classification_report(y_test, pred))

Accuracy: 0.8044692737430168
ROC-AUC: 0.8437417654808959

Confusion matrix:
 [[98 12]
 [23 46]]

Classification report:
               precision    recall  f1-score   support

           0       0.81      0.89      0.85       110
           1       0.79      0.67      0.72        69

    accuracy                           0.80       179
   macro avg       0.80      0.78      0.79       179
weighted avg       0.80      0.80      0.80       179



In [18]:
# Build a table for error analysis
errors = X_test.copy()
errors["y_true"] = y_test.values
errors["y_pred"] = pred
errors["is_error"] = (errors["y_true"] != errors["y_pred"])

errors.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,y_true,y_pred,is_error
565,3,male,24.0,2,0,24.1500,S,0,0,False
160,3,male,44.0,0,1,16.1000,S,0,0,False
553,3,male,22.0,0,0,7.2250,C,1,0,True
860,3,male,41.0,2,0,14.1083,S,0,0,False
241,3,female,NaN,1,0,15.5000,Q,1,1,False


In [19]:
# Error rate by Sex
errors.groupby("Sex")["is_error"].mean().sort_values(ascending=False)

Sex
female    0.196721
male      0.194915
Name: is_error, dtype: float64

In [20]:
# Error rate by Pclass
errors.groupby("Pclass")["is_error"].mean()

Pclass
1    0.244444
2    0.117647
3    0.200000
Name: is_error, dtype: float64

In [21]:
# Error rate by Sex + Pclass (дуже показово)
errors.groupby(["Sex", "Pclass"])["is_error"].mean().sort_values(ascending=False)

Sex     Pclass
female  3         0.357143
male    1         0.333333
        2         0.187500
        3         0.138889
female  1         0.066667
        2         0.055556
Name: is_error, dtype: float64

In [26]:
train2 = train.copy()

train2["FamilySize"] = train2["SibSp"] + train2["Parch"] + 1
train2["IsAlone"] = (train2["FamilySize"] == 1).astype(int)

feature_cols_v2 = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked", "FamilySize", "IsAlone"]
X2 = train2[feature_cols_v2]
y2 = train2["Survived"]

In [27]:
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42, stratify=y2
)

# Важливо: FamilySize і IsAlone — числові → додай їх у numeric_features
numeric_features_v2 = ["Pclass", "Age", "SibSp", "Parch", "Fare", "FamilySize", "IsAlone"]
categorical_features_v2 = ["Sex", "Embarked"]

numeric_transformer_v2 = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer_v2 = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess_v2 = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_v2, numeric_features_v2),
        ("cat", categorical_transformer_v2, categorical_features_v2)
    ]
)

model_v2 = Pipeline(steps=[
    ("preprocess", preprocess_v2),
    ("model", LogisticRegression(max_iter=2000))
])

model_v2.fit(X2_train, y2_train)

pred2 = model_v2.predict(X2_test)
proba2 = model_v2.predict_proba(X2_test)[:, 1]

acc2 = accuracy_score(y2_test, pred2)
auc2 = roc_auc_score(y2_test, proba2)

print("V2 Accuracy:", acc2)
print("V2 ROC-AUC:", auc2)
print("\nV2 Confusion matrix:\n", confusion_matrix(y2_test, pred2))

V2 Accuracy: 0.8044692737430168
V2 ROC-AUC: 0.8503293807641634

V2 Confusion matrix:
 [[97 13]
 [22 47]]


In [28]:
print("Baseline  (v1)  Acc:", acc,  "AUC:", auc)
print("Improved  (v2)  Acc:", acc2, "AUC:", auc2)
print("Delta           Acc:", acc2-acc, "AUC:", auc2-auc)

Baseline  (v1)  Acc: 0.8044692737430168 AUC: 0.8437417654808959
Improved  (v2)  Acc: 0.8044692737430168 AUC: 0.8503293807641634
Delta           Acc: 0.0 AUC: 0.006587615283267567


# ML Template: EDA → Baseline → Eval → Notes

**Goal:** (1 sentence)
**Dataset:** (where from)
**Target:** (what we predict)
**Metric:** (what we optimize)

## 1) Setup & Imports

## 2) Load Data
- Replace this part with your dataset later.
- For now we use a built-in sklearn dataset to test the template.

## 3) EDA (Exploratory Data Analysis)
- What does the data look like?
- Missing values?
- Class balance?
- Suspicious columns that could leak target?

## 4) Baseline Model
- Proper split (train/val/test or train/val)
- Baseline model + preprocessing in a Pipeline

## 5) Evaluation
- Metrics + confusion matrix
- Error analysis: where the model fails?

## 6) Notes & Next Steps
- What worked?
- What failed?
- What to try next (top 3)?

In [ ]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

import matplotlib.pyplot as plt

In [ ]:
data = load_breast_cancer(as_frame=True)
df = data.frame.copy()

# target column name in this dataset is "target"
target_col = "target"
df.head()

In [ ]:
print("Shape:", df.shape)
print("\nMissing values:\n", df.isna().sum().sort_values(ascending=False).head(10))
print("\nTarget balance:\n", df[target_col].value_counts(normalize=True))

df.describe().T.head(10)

In [ ]:
X = df.drop(columns=[target_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000))
])

model.fit(X_train, y_train)

In [ ]:
pred = model.predict(X_test)
proba = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, pred))
print("Confusion matrix:\n", confusion_matrix(y_test, pred))
print("ROC-AUC:", roc_auc_score(y_test, proba))

In [ ]:
leakage_checklist = [
    "Did I split BEFORE any stats learned from data (scaling/encoding/imputation)?",
    "Is preprocessing inside Pipeline (fit only on train)?",
    "Any columns that directly include target info (IDs, future info, labels)?",
    "Any duplicates or near-duplicates across train/test?",
]
for i, item in enumerate(leakage_checklist, 1):
    print(f"{i}. {item}")

In [4]:
import requests
r = requests.get("https://example.com", timeout=10)
print("status:", r.status_code)
print(r.text[:60])

ConnectionError: HTTPSConnectionPool(host='example.com', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7e1a23f665a0>: Failed to resolve 'example.com' ([Errno -3] Temporary failure in name resolution)"))